# Lesson 07: stdio vs. Stateless Streamable HTTP Transports

In this notebook, you will:
1. Write a clean `stdio` MCP server script that routes all logging to `sys.stderr`.
2. Test it over a real subprocess `stdio_client` pipe.
3. Build a `FastAPI` Streamable HTTP `/mcp` endpoint returning `text/event-stream` SSE progress events.

In [ ]:
!pip install -q -U mcp fastapi httpx

In [ ]:
%%writefile stdio_echo_server.py
import logging
import sys
from mcp.server.fastmcp import FastMCP

# Route all logs to stderr so stdout stays pure JSON-RPC
logging.basicConfig(stream=sys.stderr, level=logging.INFO)
mcp = FastMCP("StdioEcho")

@mcp.tool()
def echo_upper(text: str) -> str:
    """Return uppercase text."""
    logging.info("echo_upper called")
    return text.upper()

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

params = StdioServerParameters(command=sys.executable, args=["stdio_echo_server.py"])
async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        result = await session.call_tool("echo_upper", {"text": "hello mcp"})
        print("Subprocess stdio response:", result.content[0].text)

### Challenge Exercise
Create a `FastAPI` `StreamingResponse` route at `/mcp` that checks `Mcp-Method` and `Mcp-Name` headers and yields 3 `notifications/progress` SSE events before the final JSON-RPC result.